In [1]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [6]:
# Text data
# from langchain_community.document_loaders.text import TextLoader

# loader = TextLoader("data/Python.txt", encoding="utf-8")

In [7]:
# document = loader.load()

In [8]:
# document

In [9]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

In [10]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

## Ingestion Pipeline

In [16]:
# Data => Documents
import os
# from langchain_community.document_loaders.pdf import PyPDFLoader

### Documents

In [17]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [18]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 139


In [19]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

### Chunks

In [20]:
# chunks
# !pip install langchain_text_splitters

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [22]:
chunks = split_docs(all_pdf_documents)

In [23]:
len(chunks)

541

### Embedding

In [24]:
from sentence_transformers import SentenceTransformer

In [25]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [26]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("enter your hf token : ")

enter your hf token :  ········


In [27]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


### Vector Store

In [28]:
import chromadb
import uuid

In [29]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [30]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [31]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

embeddings shape: (541, 384)
total documents added in vector store= 541
docs in collection: 541


# Retrieval Pipeline

In [32]:
from sklearn.metrics.pairwise import cosine_similarity

In [33]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [34]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [35]:
rag_retriever.retrieve("Pradhan  Mantri  Matru  Vandana  Yojana  (PMMVY)??")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_7a7c893b-477c-48f5-8fe4-85783728110d',
  'document': 'Pradhan Mantri Matru Vandana Yojana or PMMVY\n It was launched in 2017 and in 2022 incorporated in Mission Shakti (umbrella \nscheme for safety, security and empowerment of women). \n Minister of Women and Child Development is the nodal implementing ministry.\n It is a conditional cash transfer scheme for Pregnant Women & Lactating \nMothers (PW&LM). \n Beneficiaries: PW&LM of at least 19 years of age in informal and unorganized \nsectors.\n Benefits:',
  'metadata': {'trapped': '/False',
   'producer': 'Adobe PDF Library 17.0',
   'content_length': 447,
   'creator': 'Adobe InDesign 19.0 (Windows)',
   'page': 22,
   'doc_index': 85,
   'source': 'data/pdfs\\4d653-government-schemes-in-news.pdf',
   'creationdate': '2024-04-20T19:26:45+05:30',
   'moddate': '2024-04-20T19:27:17+05:30',
   'total_pages': 134,
   'page_label': '23'},
  'distance': 0.38992273807525635,
  'similarity_score': 0.6100772619247437,
  'rank': 1

# Integrate with LLMs

# Groq-AI

In [36]:
# !pip install langchain-groq

In [ ]:
import os
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key = "groq_api_key",
    model = "openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=1024
)

In [39]:
# generate our retrieval-augmented output

def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context :
        print("We found no relevant context for the given query")

    # context query
    prompt = f"""use given context to generate the answer for the query if not match do not do hallucination
                 Context : {context}
                 Query : {query} """
    response = llm.invoke(prompt) # expecting a list as prompt
    return response.content

In [40]:
answer = generate_output("Pradhan  Mantri  Matru  Vandana  Yojana  (PMMVY)??", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


In [41]:
print(answer)

**Pradhan Mantri Matru Vandana Yojana (PMMVY)**  

- **What it is:** A conditional cash‑transfer scheme (formerly called the *Maternity Benefit Programme*) that provides financial assistance to pregnant women and lactating mothers.  

- **Launch & Integration:**  
  - Launched in **2017**.  
  - In **2022** it was incorporated under **Mission Shakti**, the umbrella scheme for the safety, security and empowerment of women.  

- **Implementing Ministry:** **Ministry of Women and Child Development** (the nodal ministry).  

- **Target Beneficiaries:**  
  - Pregnant women and lactating mothers (PW & LM) who are **at least 19 years old**.  
  - Specifically those working in the **informal and unorganized sectors**.  

- **Primary Objective / Benefits:**  
  - Provides cash incentives to improve **health and nutrition** of pregnant and nursing mothers.  
  - Aims to create a better enabling environment for **women’s empowerment**.  

In short, PMMVY is a government‑run cash‑benefit program 